<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase37_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 37 — Synthetic Procedure provenance and Arm A source-readiness gate
**Google Drive → Google Drive · no Athena needed · no invented medical evidence**

**Why:** Phase 36 independently reconciled 148 observations to the original 19-person synthetic FHIR source. That source has **38 ImagingStudy records and zero Procedure records**. This phase produces **38 expressly manufactured synthetic test Procedure fixtures**, with one-to-one study linkage and corresponding synthetic Provenance, while leaving all original observations untouched. No earlier Phase 29 Procedures are copied into this different cohort. It also updates a source-only Arm A readiness matrix and creates a private 19-key Athena review worksheet, without assigning concept IDs.

**Input integrity:** Requires SHA-matched Phase 36 SHAREABLE and EXPERIMENT_CONTRACT and the **original exact-SHA Phase 33 FHIR bundle**. Phase 36 SOURCE_FHIR_CROSSWALK and TARGET_FIELD_EVIDENCE are required and checked against its report hashes. Variants ending `(1)` are supported; mixed-run files are rejected. The Phase 36 internal protocol is preserved unchanged. Change `DRIVE_RELATIVE_FOLDER` if needed.

**Output limitations:** Manufactured `Procedure` and `Provenance` are *synthetic test fixtures generated now*, not original source records or real patient history. Original clinical Procedure provenance and DICOM attestation remain unavailable. The official Athena/Mi-CDM Arm A, pyOMOP execution, MCI cohort equivalence, original TFL C, and four-arm evaluation **remain unexecuted**. No source gender, race, ethnicity or birthdate is inferred.

In [1]:
import os
DRIVE_ROOT_OVERRIDE=os.environ.get('ICHI2027_DRIVE_ROOT','').strip()
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json,hashlib,csv,copy,re,os,datetime,uuid
from pathlib import Path
from collections import Counter,defaultdict
DRIVE_ROOT=Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
DRIVE_RELATIVE_FOLDER=''   # Set e.g. 'NeuroFHIR_ICHI2027' if there are several projects
ROOT=DRIVE_ROOT/DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
assert ROOT.is_dir(),f'Google Drive folder not found: {ROOT}'
IMAGEWG='8f0a19bb94e941bb335a097bc9851fab672621c7'
OMOP55='4a910305b2cb74a4fc2b2c34baf44eb0542ff03f'
EXPECTED_PHASE='36_independent_original_fhir_crosswalk_and_standards_arm_gate'

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def canon(v):return json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False)
def name_match(real,wanted):
    s,e=os.path.splitext(wanted)
    return real==wanted or bool(re.fullmatch(re.escape(s)+r'\(\d+\)'+re.escape(e),real))
def walk(root):
    for d,ds,fs in os.walk(str(root),onerror=lambda err:None):
        ds[:]=[x for x in ds if not x.startswith('.') and x not in {'_PRIVATE_ATHENA','node_modules','.git'}]
        for f in sorted(fs):yield Path(d)/f
def locate_by_hash(digest,basename,folder=None):
    for p in (folder.iterdir() if folder else walk(ROOT)):
        try:
            if p.is_file() and name_match(p.name,basename) and sha(p)==digest:return p
        except OSError:pass
    return None
def csvread(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def csvwrite(path,rows,columns):
    with Path(path).open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=columns,extrasaction='ignore');writer.writeheader();writer.writerows(rows)
def jsonwrite(path,obj):Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')

candidates=[]
for p in walk(ROOT):
    if not name_match(p.name,'ICHI2027_Phase36_SHAREABLE.json'):continue
    try:
        rep=json.loads(p.read_text(encoding='utf-8'))
        if rep.get('phase')!=EXPECTED_PHASE:continue
        contract=locate_by_hash(rep['public_output_sha256']['experiment_contract'],'ICHI2027_Phase36_EXPERIMENT_CONTRACT.json',p.parent)
        crosswalk=locate_by_hash(rep['public_output_sha256']['source_fhir_crosswalk'],'ICHI2027_Phase36_SOURCE_FHIR_CROSSWALK.csv',p.parent)
        fieldaudit=locate_by_hash(rep['public_output_sha256']['target_field_evidence'],'ICHI2027_Phase36_TARGET_FIELD_EVIDENCE.csv',p.parent)
        fhir=locate_by_hash(rep['upstream']['phase33_original_fhir_sha256'],'ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json')
        if all((contract,crosswalk,fieldaudit,fhir)):
            candidates.append((p,rep,contract,crosswalk,fieldaudit,fhir));break
    except (ValueError,KeyError,OSError):continue
assert candidates, 'Missing exact matched Phase36 SHAREABLE, CONTRACT, CROSSWALK, TARGET_FIELD_EVIDENCE and original Phase33 FHIR. Copy all required original Colab files to Drive, without renaming their contents.'
J36,P36,CONTRACT_PATH,CROSSWALK_PATH,TARGET_FIELDS_PATH,FHIR_PATH=candidates[0]
P36_CONTRACT=json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
assert P36_CONTRACT['protocol_id']=='ICHI2027_P36_FOUR_ARM_FREEZE_V1'
assert P36_CONTRACT['status']=='INTERNAL_VERSIONED_FUTURE_PROTOCOL_NOT_PUBLICLY_PREREGISTERED'
assert P36_CONTRACT['registration_boundary']['original_fhir_bundle_sha256']==sha(FHIR_PATH)
assert P36['checks']=={'total':17,'passing':17}
assert P36['upstream']['imagewg_commit']==IMAGEWG and P36['upstream']['omop55_commit']==OMOP55
assert P36['independent_source_audit']['rows_exactly_reconciled']==148
CROSSWALK=csvread(CROSSWALK_PATH)
assert len(CROSSWALK)==148 and len(set(r['source_observation_ref'] for r in CROSSWALK))==148
assert all(r['status']=='PASS' and int(r['mismatch_count'])==0 and int(r['field_checks'])==18 for r in CROSSWALK)
FIELD_AUDIT=csvread(TARGET_FIELDS_PATH)
assert any(r['table']=='image_occurrence' and r['field']=='procedure_occurrence_id' and r['evidence_status']=='BLOCKED_NO_PHASE33_SOURCE_PROCEDURE' for r in FIELD_AUDIT)
FHIR_ORIG=json.loads(FHIR_PATH.read_text(encoding='utf-8'))
assert FHIR_ORIG.get('resourceType')=='Bundle' and FHIR_ORIG.get('type')=='collection'
OUT=J36.parent
print('Verified Phase36 report:',J36.name)
print('Verified frozen experiment contract SHA:',sha(CONTRACT_PATH))
print('Verified untouched Phase33 source SHA:',sha(FHIR_PATH))
print('Output Google Drive folder:',OUT)

Verified Phase36 report: ICHI2027_Phase36_SHAREABLE.json
Verified frozen experiment contract SHA: 00d53ee91a60adf970e28b18b8db101c744b13c3e8f64e8cfdafa99cfb8a72f0
Verified untouched Phase33 source SHA: 4d2e64b3dad926f9a32fc2d243f8ebb04dcff0159d60c5343f01e0a1985d1bf4
Output Google Drive folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


## 1 · Reconcile exact source references and manufacture explicitly labeled fixtures
Each newly manufactured synthetic Procedure is derived **for testing only** from the timestamp, subject and identity of one original synthetic ImagingStudy. It is not a claim that a source Procedure existed. The provenance timestamp records this notebook's actual fixture-generation run, not the date of the synthetic MRI. Existing FHIR entries are copied without modification, except a synthetic `ImagingStudy.procedureReference` is added to the copied study. All original source bytes stay unchanged on Drive.

In [3]:
ENTITY=defaultdict(dict)
for e in FHIR_ORIG['entry']:
    r=e['resource'];k=r['resourceType'];ident=r['id']
    assert ident not in ENTITY[k],f'Duplicate original source resource {k}/{ident}'
    ENTITY[k][ident]=r
assert {k:len(ENTITY[k]) for k in ['Patient','Device','ImagingStudy','Observation','Provenance']}=={'Patient':19,'Device':19,'ImagingStudy':38,'Observation':148,'Provenance':148}
assert len(ENTITY.get('Procedure',{}))==0
assert all('procedureReference' not in st for st in ENTITY['ImagingStudy'].values())
assert all('gender' not in p and 'birthDate' not in p for p in ENTITY['Patient'].values())
all_studies=sorted(ENTITY['ImagingStudy'].values(),key=lambda x:x['id'])
all_obs=sorted(ENTITY['Observation'].values(),key=lambda x:x['id'])
assert all(len(s.get('series',[]))==1 for s in all_studies)
SOURCE_REF='https://example.org/ichi2027/phase37/synthetic'
FIXTURE_TAG={'system':SOURCE_REF+'/fixture-kind','code':'manufactured-synthetic-source-linked-test'}
FIXTURE_PROV_TAG={'system':SOURCE_REF+'/fixture-kind','code':'manufactured-synthetic-procedure-provenance'}
SOURCE_IDS={x['id'] for x in all_studies}
assert len(SOURCE_IDS)==38
orig_refs={r['source_observation_ref'] for r in CROSSWALK}
assert orig_refs=={'Observation/'+o['id'] for o in all_obs}

def uid_for(study_id):
    # UUIDv5 creates a stable fixture ID; never confuse it with real DICOM evidence.
    return uuid.uuid5(uuid.NAMESPACE_URL,'ICHI2027/phase37/synthetic-procedure/'+study_id).hex[:24]

def make_bundle(src,run_utc):
    b=copy.deepcopy(src)
    existing=defaultdict(dict)
    for e in b['entry']:
        r=e['resource'];existing[r['resourceType']][r['id']]=r
    dev_by_person={('Patient/'+x['id'].removeprefix('dev-')):x for x in existing['Device'].values()}
    # Device patient linking is checked explicitly, not assumed to encode clinical identity.
    for o in existing['Observation'].values():
        person=o['subject']['reference'];device=o['device']['reference']
        assert device.startswith('Device/') and device.split('/',1)[1] in existing['Device']
        assert existing['ImagingStudy'][o['focus'][0]['reference'].split('/',1)[1]]['subject']['reference']==person
    added=[]
    for sid in sorted(existing['ImagingStudy']):
        st=existing['ImagingStudy'][sid]
        person=st['subject']['reference'];start=st['started']
        assert person.split('/',1)[1] in existing['Patient']
        procid='synth37-proc-'+uid_for(sid)
        provid='synth37-prov-'+uid_for(sid)
        st['procedureReference']={'reference':'Procedure/'+procid}
        procedure={'resourceType':'Procedure','id':procid,'status':'completed',
            'meta':{'tag':[copy.deepcopy(FIXTURE_TAG)]},
            'identifier':[{'system':SOURCE_REF+'/procedure-fixture','value':procid}],
            'code':{'coding':[{'system':SOURCE_REF+'/procedure-code','code':'SYN-BRAIN-MRI','display':'Manufactured synthetic MRI procedure test fixture'}]},
            'subject':{'reference':person},'performedDateTime':start,
            'note':[{'text':'Post-hoc manufactured synthetic test fixture from an existing synthetic ImagingStudy. NOT source-attested clinical Procedure.'}]}
        related_obs=[o for o in existing['Observation'].values() if o.get('focus',[{}])[0].get('reference')=='ImagingStudy/'+sid]
        devrefs={o['device']['reference'] for o in related_obs}
        assert len(devrefs)==1,f'Device ambiguity in source synthetic study {sid}'
        devref=next(iter(devrefs))
        provenance={'resourceType':'Provenance','id':provid,'recorded':run_utc,
            'meta':{'tag':[copy.deepcopy(FIXTURE_PROV_TAG)]},
            'target':[{'reference':'Procedure/'+procid}],
            'entity':[{'role':'derivation','what':{'reference':'ImagingStudy/'+sid}}],
            'agent':[{'type':{'text':'synthetic fixture generator; not a clinical agent'},'who':{'reference':devref}}],
            'extension':[{'url':SOURCE_REF+'/StructureDefinition/manufactured-fixture','valueBoolean':True}]}
        b['entry'].extend([{'resource':procedure},{'resource':provenance}])
        added.append({'person_ref':person,'study_ref':'ImagingStudy/'+sid,'procedure_ref':'Procedure/'+procid,
            'provenance_ref':'Provenance/'+provid,'device_ref':devref,'study_started':start,
            'fixture_kind':'POST_HOC_MANUFACTURED_SYNTHETIC_ONLY','clinical_source_procedure_attested':'FALSE'})
    return b,added

# One UTC timestamp per full run. Reusing the timestamp must reproduce identical content.
RUN_UTC=datetime.datetime.now(datetime.timezone.utc).isoformat()
PATCHED,LINKS=make_bundle(FHIR_ORIG,RUN_UTC)
assert len(LINKS)==38
assert canon(make_bundle(FHIR_ORIG,RUN_UTC)[0])==canon(PATCHED), 'In-memory idempotence failed'
assert sha(FHIR_PATH)==P36['upstream']['phase33_original_fhir_sha256'], 'Original Phase33 file was unexpectedly modified'
print('Manufactured synthetic fixtures:',len(LINKS),'new Procedures and',len(LINKS),'new corresponding Provenance records')

Manufactured synthetic fixtures: 38 new Procedures and 38 new corresponding Provenance records


## 2 · Independent source-to-fixture integrity and deliberate negative controls
Independent checks reject forged clinical status, wrong-person links, changed acquisition dates, missing or duplicate Procedure links, orphan provenance, and any change to the source's 148 observations. The original Phase 36 provenance audit is *not* retroactively rewritten.

In [4]:
def verify_patch(original,candidate,expected_time):
    errs=[];O=defaultdict(dict);X=defaultdict(dict)
    for name,b,index in [('original',original,O),('patched',candidate,X)]:
        for e in b.get('entry',[]):
            r=e.get('resource',{});k=r.get('resourceType');i=r.get('id')
            if not k or not i or i in index[k]:errs.append(name+'_MISSING_OR_DUPLICATE_RESOURCE_ID')
            else:index[k][i]=r
    for k,count in [('Patient',19),('Device',19),('ImagingStudy',38),('Observation',148),('Procedure',38),('Provenance',186)]:
        if len(X[k])!=count:errs.append('BAD_'+k+'_COUNT')
    for k in ('Patient','Device','Observation'):
        if set(O[k])!=set(X[k]) or any(canon(O[k][i])!=canon(X[k].get(i)) for i in O[k]):errs.append('ORIGINAL_'+k+'_CHANGED')
    if any(canon(O['Provenance'][i])!=canon(X['Provenance'].get(i)) for i in O['Provenance']):errs.append('ORIGINAL_PROVENANCE_CHANGED')
    seen=set()
    for sid,source in O['ImagingStudy'].items():
        target=X['ImagingStudy'].get(sid,{})
        reverted=copy.deepcopy(target);ref=reverted.pop('procedureReference',{})
        if canon(reverted)!=canon(source):errs.append('ORIGINAL_STUDY_FIELDS_CHANGED')
        pid=ref.get('reference','').removeprefix('Procedure/')
        if not pid or pid in seen or pid not in X['Procedure']:errs.append('MISSING_OR_DUPLICATE_PROCEDURE_LINK');continue
        seen.add(pid)
        p=X['Procedure'][pid]
        if not (p.get('subject',{}).get('reference')==source['subject']['reference'] and
                p.get('performedDateTime')==source['started']):errs.append('PROCEDURE_PERSON_OR_DATE_MISMATCH')
        if FIXTURE_TAG not in p.get('meta',{}).get('tag',[]):errs.append('UNMARKED_PROCEDURE_FIXTURE')
        if p.get('code',{}).get('coding',[{}])[0].get('code')!='SYN-BRAIN-MRI':errs.append('WRONG_TEST_FIXTURE_CODE')
        provs=[r for k,r in X['Provenance'].items() if k not in O['Provenance'] and any(t.get('reference')=='Procedure/'+pid for t in r.get('target',[]))]
        if len(provs)!=1:errs.append('MISSING_OR_DUPLICATE_SYNTHETIC_PROVENANCE');continue
        pr=provs[0]
        if FIXTURE_PROV_TAG not in pr.get('meta',{}).get('tag',[]):errs.append('UNMARKED_SYNTHETIC_PROVENANCE')
        if pr.get('recorded')!=expected_time:errs.append('WRONG_FIXTURE_GENERATION_TIME')
        if not any(e.get('what',{}).get('reference')=='ImagingStudy/'+sid for e in pr.get('entity',[])):errs.append('WRONG_PROVENANCE_STUDY')
        if not any(ex.get('valueBoolean') is True and ex.get('url','').endswith('/manufactured-fixture') for ex in pr.get('extension',[])):errs.append('SYNTHETIC_FIXTURE_FLAG_MISSING')
        expected_devices={o['device']['reference'] for o in O['Observation'].values() if o.get('focus',[{}])[0].get('reference')=='ImagingStudy/'+sid}
        if len(expected_devices)!=1 or not any(a.get('who',{}).get('reference') in expected_devices for a in pr.get('agent',[])):errs.append('PROVENANCE_DEVICE_MISMATCH')
    if len(seen)!=38 or seen!=set(X['Procedure']):errs.append('ORPHAN_PROCEDURE')
    added_prov={k for k in X['Provenance'] if k not in O['Provenance']}
    if len(added_prov)!=38:errs.append('ORPHAN_OR_MISSING_PROVENANCE')
    for k in added_prov:
        pr=X['Provenance'][k]
        if len(pr.get('target',[]))!=1 or pr['target'][0].get('reference','').removeprefix('Procedure/') not in seen:errs.append('ORPHAN_SYNTHETIC_PROVENANCE')
    if any('Procedure/' in canon(O['ImagingStudy'][s]) for s in O['ImagingStudy']):errs.append('SOURCE_ALREADY_HAD_PROCEDURE')
    return sorted(set(errs))

assert verify_patch(FHIR_ORIG,PATCHED,RUN_UTC)==[],verify_patch(FHIR_ORIG,PATCHED,RUN_UTC)
NEG=[]
def negative(label,mutation,expected):
    x=copy.deepcopy(PATCHED);mutation(x);errors=verify_patch(FHIR_ORIG,x,RUN_UTC)
    NEG.append({'test':label,'expected_error':expected,'observed_errors':'|'.join(errors),'detected':expected in errors})
    assert NEG[-1]['detected'],f'{label} undetected: {errors}'
def idx(b,kind):return [e['resource'] for e in b['entry'] if e['resource']['resourceType']==kind]
def first(b,kind):return idx(b,kind)[0]
negative('wrong_person_on_manufactured_procedure',lambda b:first(b,'Procedure')['subject'].update({'reference':'Patient/held33-p019'}),'PROCEDURE_PERSON_OR_DATE_MISMATCH')
negative('wrong_acquisition_date',lambda b:first(b,'Procedure').update({'performedDateTime':'1999-01-01T00:00:00Z'}),'PROCEDURE_PERSON_OR_DATE_MISMATCH')
negative('remove_fixture_tag',lambda b:first(b,'Procedure')['meta'].update({'tag':[]}), 'UNMARKED_PROCEDURE_FIXTURE')
negative('replace_fixture_with_clinical_label',lambda b:first(b,'Procedure')['code']['coding'][0].update({'code':'CLINICAL-MRI'}),'WRONG_TEST_FIXTURE_CODE')
negative('broken_study_link',lambda b:first(b,'ImagingStudy').update({'procedureReference':{'reference':'Procedure/absent'}}),'MISSING_OR_DUPLICATE_PROCEDURE_LINK')
negative('duplicate_study_link',lambda b:idx(b,'ImagingStudy')[1].update({'procedureReference':first(b,'ImagingStudy')['procedureReference']}),'MISSING_OR_DUPLICATE_PROCEDURE_LINK')
negative('mutated_original_observation',lambda b:first(b,'Observation')['valueQuantity'].update({'value':0}),'ORIGINAL_Observation_CHANGED')
negative('mutated_original_study_uid',lambda b:first(b,'ImagingStudy')['identifier'][0].update({'value':'FAKE'}),'ORIGINAL_STUDY_FIELDS_CHANGED')
negative('remove_manufactured_provenance',lambda b:b['entry'].remove(next(e for e in b['entry'] if e['resource']['resourceType']=='Provenance' and e['resource']['id'].startswith('synth37-prov-'))),'MISSING_OR_DUPLICATE_SYNTHETIC_PROVENANCE')
negative('wrong_provenance_study',lambda b:next(e['resource'] for e in b['entry'] if e['resource']['resourceType']=='Provenance' and e['resource']['id'].startswith('synth37-prov-'))['entity'][0]['what'].update({'reference':'ImagingStudy/bogus'}),'WRONG_PROVENANCE_STUDY')
negative('wrong_fixture_generation_time',lambda b:next(e['resource'] for e in b['entry'] if e['resource']['resourceType']=='Provenance' and e['resource']['id'].startswith('synth37-prov-')).update({'recorded':'2000-01-01T00:00:00Z'}),'WRONG_FIXTURE_GENERATION_TIME')
negative('claim_not_synthetic',lambda b:next(e['resource'] for e in b['entry'] if e['resource']['resourceType']=='Provenance' and e['resource']['id'].startswith('synth37-prov-'))['extension'][0].update({'valueBoolean':False}),'SYNTHETIC_FIXTURE_FLAG_MISSING')
# Duplicated patched resource must be detected without using original source lookups.
x=copy.deepcopy(PATCHED);x['entry'].append(copy.deepcopy(next(e for e in x['entry'] if e['resource']['resourceType']=='Procedure')))
e=verify_patch(FHIR_ORIG,x,RUN_UTC);NEG.append({'test':'duplicate_procedure_resource','expected_error':'patched_MISSING_OR_DUPLICATE_RESOURCE_ID','observed_errors':'|'.join(e),'detected':'patched_MISSING_OR_DUPLICATE_RESOURCE_ID' in e});assert NEG[-1]['detected']
assert len(NEG)==13 and all(r['detected'] for r in NEG)
print('Exact source + synthetic fixture integrity: PASS; corrupted test cases detected:',len(NEG),'/',len(NEG))

Exact source + synthetic fixture integrity: PASS; corrupted test cases detected: 13 / 13


## 3 · Update only evidence status, not official OMOP fields or claims
No numeric OMOP concept IDs are selected. The procedure linkage gap can be closed *only for explicitly manufactured synthetic test cases*, not for real data, and the 36 older vocabulary-related gap instances stay blocked. Source gender/birthdate remain absent. A source-to-fixture diagnostic join is generated for all 148 observations, but no rows are written into an OMOP or MI-CDM target.

In [5]:
# Observations -> studies -> synthetic fixture: a source-side shadow diagnostic, NOT OMOP load.
JOIN=[]
link_by_study={r['study_ref']:r for r in LINKS}
for obs in all_obs:
    study_ref=obs['focus'][0]['reference'];l=link_by_study[study_ref]
    JOIN.append({'observation_ref':'Observation/'+obs['id'],'person_ref':obs['subject']['reference'],
        'study_ref':study_ref,'synthetic_procedure_ref':l['procedure_ref'],
        'synthetic_procedure_provenance_ref':l['provenance_ref'],
        'join_status':'EXACT_SYNTHETIC_FIXTURE_ONLY','official_target_loaded':'FALSE'})
assert len(JOIN)==148 and len({r['observation_ref'] for r in JOIN})==148
assert len({r['synthetic_procedure_ref'] for r in JOIN})==38
assert all(r['person_ref']==next(k['person_ref'] for k in LINKS if k['study_ref']==r['study_ref']) for r in JOIN)
UPDATED=[]
for old in FIELD_AUDIT:
    row=dict(old)
    if row['table']=='image_occurrence' and row['field']=='procedure_occurrence_id':
        row['evidence_status']='SYNTHETIC_MANUFACTURED_PROCEDURE_LINK_ONLY_NOT_SOURCE_ATTESTED'
    elif row['table']=='procedure_occurrence' and row['field'] in ('procedure_occurrence_id','person_id','procedure_date'):
        row['evidence_status']='SYNTHETIC_MANUFACTURED_PROCEDURE_FIXTURE_ONLY'
    elif row['table']=='procedure_occurrence' and row['field'] in ('procedure_concept_id','procedure_type_concept_id'):
        row['evidence_status']='SYNTHETIC_PROCEDURE_FIXTURE_READY_VOCABULARY_STILL_BLOCKED'
    row['synthetic_source_cohort']='PHASE33_19_PERSONS_PLUS_38_POSTHOC_SYNTHETIC_PROCEDURE_FIXTURES'
    row['armA_field_populated']='False' # No actual target materialization.
    UPDATED.append(row)
assert len(UPDATED)==len(FIELD_AUDIT)
assert all(row['armA_field_populated']=='False' for row in UPDATED)
BASE_BINDING_KEYS=['IO_MODALITY_MR','IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT',
  'IF_TYPE_ALGORITHM','PERSON_GENDER_F','PERSON_GENDER_M','PERSON_RACE_UNRECORDED',
  'PERSON_ETHNICITY_UNRECORDED','PROCEDURE_MRI','PROCEDURE_TYPE_SYN','MEAS_HIP_LEFT',
  'MEAS_HIP_RIGHT','MEAS_ENT_LEFT','MEAS_ENT_RIGHT','MEAS_TYPE_SYN','UNIT_MM3','IF_EVENT_MEAS_ID']
assert len(BASE_BINDING_KEYS)==19 and len(set(BASE_BINDING_KEYS))==19
GAPS=[
 {'gate':'Phase33 source Procedure evidence','before':'0 Procedure / 38 unlinked synthetic ImagingStudy','after':'38 POST-HOC MANUFACTURED SYNTHETIC FIXTURE links','clinical_status':'NOT_SOURCE_ATTESTED'},
 {'gate':'Phase33 raw original Observations','before':'148','after':'148 unchanged','clinical_status':'SYNTHETIC_SOURCE_ONLY'},
 {'gate':'Phase33 source demographics','before':'Gender and birthDate missing in 19 source Patient records','after':'Still missing; no imputation','clinical_status':'BLOCKED_REAL_SOURCE_EVIDENCE'},
 {'gate':'Original Phase29 vocabulary gap instances','before':'36 unresolved','after':'36 unresolved; not silently transferred to new cohort','clinical_status':'BLOCKED_VOCABULARY'},
 {'gate':'Independent frozen Athena 19 binding decisions','before':'0 source-supported approvals reported','after':'0 automatically approved','clinical_status':'REQUIRES_INDEPENDENT_REVIEW'},
 {'gate':'IMAGE_FEATURE event-field linkage metadata','before':'Unvalidated','after':'Unvalidated; retain sidecar until independently adjudicated','clinical_status':'BLOCKED_METADATA'},
 {'gate':'Official MI-CDM + OMOP 5.5 target load','before':'Not executed','after':'Not executed','clinical_status':'BLOCKED'},
 {'gate':'Source-proxy versus clinical MCI cohort','before':'No attested MCI','after':'No attested MCI','clinical_status':'NOT_EVALUABLE'}]

## 4 · Private Athena review intake (never published)
The notebook creates `_PRIVATE_ATHENA/PHASE37_19_BINDING_REVIEW_TEMPLATE.csv` and `_PRIVATE_ATHENA/PHASE37_VOCAB_SNAPSHOT_MANIFEST_TEMPLATE.json` **only if they do not already exist**. This phase does not read or export concept identifiers or reviewer names into public outputs, and the presence of an export alone never approves a mapping. Fill the template with an authorized frozen Athena `CONCEPT.csv` and independently reviewed source-based decisions before building Arm A in a subsequent phase.

In [6]:
PRIVATE=OUT/'_PRIVATE_ATHENA';PRIVATE.mkdir(parents=True,exist_ok=True)
REVIEW=PRIVATE/'PHASE37_19_BINDING_REVIEW_TEMPLATE.csv'
if not REVIEW.exists():
    rows=[]
    for k in BASE_BINDING_KEYS:
        source_policy=('NOT_APPLICABLE_SOURCE_GENDER_ABSENT' if k.startswith('PERSON_GENDER_') else
                       'REVIEW_MISSINGNESS_POLICY' if k.startswith('PERSON_RACE_') or k.startswith('PERSON_ETHNICITY_') else
                       'REQUIRES_EXPLICIT_EVENT_FIELD_ADJUDICATION' if k=='IF_EVENT_MEAS_ID' else
                       'SYNTHETIC_FIXTURE_NOT_CLINICALLY_ATTESTED' if k.startswith('PROCEDURE_') else 'REQUIRES_SOURCE_SUPPORTED_REVIEW')
        rows.append({'binding_key':k,'proposed_concept_id':'','snapshot_sha256':'',
            'reviewer':'','review_date_utc':'','independent_review_evidence':'',
            'exact_source_code_and_definition':'','semantic_and_domain_justification':'',
            'source_application_policy':source_policy,'independent_review_decision':'PENDING'})
    csvwrite(REVIEW,rows,list(rows[0]))
MANIFEST=PRIVATE/'PHASE37_VOCAB_SNAPSHOT_MANIFEST_TEMPLATE.json'
if not MANIFEST.exists():jsonwrite(MANIFEST,{
    'private_authorized_CONCEPT_file':'CONCEPT.csv or CONCEPT.csv.gz','expected_sha256':'',
    'athena_release_or_export_date':'','source_or_license_notes':'',
    'note':'Never commit this file or reviewed concept IDs to public GitHub. The template alone never approves bindings.'})
PRIVATE_GUIDE=PRIVATE/'PHASE37_README_PRIVATE.txt'
if not PRIVATE_GUIDE.exists():PRIVATE_GUIDE.write_text('Private: store authorized frozen Athena CONCEPT.csv and independently reviewed 19-key bindings here.\nDo not upload the _PRIVATE_ATHENA folder to GitHub or publicly share concept IDs/reviewer data.\nPhase33 source lacks gender and birthdate; no vocabulary can invent them.\nAn event-field concept needs independent ImageWG field adjudication.\n',encoding='utf-8')
assert len(csvread(REVIEW))==19
assert all(not r['proposed_concept_id'].strip() for r in csvread(REVIEW)) if all(r['independent_review_decision']=='PENDING' for r in csvread(REVIEW)) else True
print('Created/retained private 19-key worksheet and snapshot manifest:',PRIVATE)

Created/retained private 19-key worksheet and snapshot manifest: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/_PRIVATE_ATHENA


## 5 · Write the public-safe outputs and verify all files
All public files contain only synthetic fixture identifiers, evidence status, and hashes. The original input files and frozen Phase 36 contract stay unchanged. The new synthetic bundle has source resource copies plus 38 linked test Procedures and 38 newly generated synthetic Provenance records. The bundle itself remains fully synthetic and is suitable for a synthetic-only GitHub repo; never push the private vocabulary folder.

In [7]:
assert sha(FHIR_PATH)==P36['upstream']['phase33_original_fhir_sha256']
assert sha(CONTRACT_PATH)==P36['public_output_sha256']['experiment_contract']
OUTFILES={
 'ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS.csv':(LINKS,list(LINKS[0])),
 'ICHI2027_Phase37_SOURCE_TO_FIXTURE_JOIN.csv':(JOIN,list(JOIN[0])),
 'ICHI2027_Phase37_NEGATIVE_LINK_TESTS.csv':(NEG,list(NEG[0])),
 'ICHI2027_Phase37_ARM_A_FIELD_EVIDENCE.csv':(UPDATED,list(UPDATED[0])),
 'ICHI2027_Phase37_UNRESOLVED_GATES.csv':(GAPS,list(GAPS[0]))}
for filename,(rows,fields) in OUTFILES.items():csvwrite(OUT/filename,rows,fields)
BUNDLE_PATH=OUT/'ICHI2027_Phase37_SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json'
jsonwrite(BUNDLE_PATH,PATCHED)
assert verify_patch(FHIR_ORIG,json.loads(BUNDLE_PATH.read_text(encoding='utf-8')),RUN_UTC)==[]
assert sha(CONTRACT_PATH)==P36['public_output_sha256']['experiment_contract']
assert sha(FHIR_PATH)==P36['upstream']['phase33_original_fhir_sha256']
REPORT={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'37_posthoc_manufactured_synthetic_procedures_for_phase33_cohort',
 'run_utc':RUN_UTC,
 'privacy':'Fully synthetic/public-safe. Private Athena files and reviewer data excluded from all public outputs; no real clinical Procedure or DICOM evidence.',
 'upstream':{'phase36_shareable_sha256':sha(J36),'phase36_contract_sha256':sha(CONTRACT_PATH),
   'phase33_original_fhir_sha256':sha(FHIR_PATH),'imagewg_commit':IMAGEWG,'omop55_commit':OMOP55},
 'source':{'persons':19,'imaging_studies':38,'observations':148,'original_source_procedure_count':0,
   'original_source_provenance_count':148,'gender_attested':False,'birthdate_attested':False,
   'source_MCI_status_attested':False,'real_DICOM_UID_provenance_attested':False},
 'manufactured_synthetic_test_fixture':{
   'new_procedures':38,'new_procedure_provenance':38,'studies_with_1_to_1_fixture_links':38,
   'diagnostic_observation_to_fixture_joins':148,'source_Observations_unchanged':True,
   'original_source_File_sha256_unchanged':True,'independent_clinical_source_procedures_created':0,
   'in_memory_idempotence_at_fixed_run_timestamp':True},
 'tests':{'negative_link_cases_total':len(NEG),'negative_link_cases_detected':sum(int(x['detected']) for x in NEG),
   'positive_source_fixture_integrity_verified':True,'phase36_source_crosswalk_verified':True},
 'gaps':{'original_phase29_vocab_gaps_blocked':36,'original_athena_binding_keys':19,
   'new_concept_ids_automatically_selected':0,'event_field_metadata_independently_adjudicated':False,
   'real_source_procedure_gap_resolved':False,'synthetic_phase33_procedure_fixture_gap_only_addressed':True},
 'gates':{'synthetic_fixture_provenance_links_verified':True,'faithful_armA_validated':False,
   'official_MI_CDM_loaded':False,'pyOMOP_ETL_executed':False,'original_generic_armC_executed':False,
   'true_four_arm_comparison_completed':False,'official_target_cohort_equivalence_tested':False,
   'independent_blinding_completed':False,'real_MRI_rows_processed':0},
 'artifact_sha256':{**{f:sha(OUT/f) for f in OUTFILES},BUNDLE_PATH.name:sha(BUNDLE_PATH)},
 'next_gate':'Obtain and freeze an authorized private Athena CONCEPT snapshot; independently source-check 19 bindings and adjudicate IMAGE_FEATURE event-field metadata. Do not assign Phase33 gender or birthdates. Only then create official-target-syntactic Arm A staging for these synthetic fixtures with exact required-field and SQL/idempotence tests. Real clinical Procedure and DICOM evidence remain separate.'}
REPORT_PATH=OUT/'ICHI2027_Phase37_SHAREABLE.json'
jsonwrite(REPORT_PATH,REPORT)
assert all(sha(OUT/n)==digest for n,digest in REPORT['artifact_sha256'].items())
assert len(csvread(OUT/'ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS.csv'))==38
assert len(csvread(OUT/'ICHI2027_Phase37_SOURCE_TO_FIXTURE_JOIN.csv'))==148
assert all(r['detected']=='True' for r in csvread(OUT/'ICHI2027_Phase37_NEGATIVE_LINK_TESTS.csv'))
print('PHASE37 PASS: 38/38 synthetic Procedure + Provenance links, 148/148 diagnostic joins,',len(NEG),'/',len(NEG),'negative controls.')
print('Files: SHAREABLE.json, SYNTHETIC_PROCEDURE_LINKS.csv, SOURCE_TO_FIXTURE_JOIN.csv, NEGATIVE_LINK_TESTS.csv, ARM_A_FIELD_EVIDENCE.csv, UNRESOLVED_GATES.csv, SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json')
print('UPLOAD NEXT: ICHI2027_Phase37_SHAREABLE.json; ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS.csv; ICHI2027_Phase37_NEGATIVE_LINK_TESTS.csv; ICHI2027_Phase37_UNRESOLVED_GATES.csv')

PHASE37 PASS: 38/38 synthetic Procedure + Provenance links, 148/148 diagnostic joins, 13 / 13 negative controls.
Files: SHAREABLE.json, SYNTHETIC_PROCEDURE_LINKS.csv, SOURCE_TO_FIXTURE_JOIN.csv, NEGATIVE_LINK_TESTS.csv, ARM_A_FIELD_EVIDENCE.csv, UNRESOLVED_GATES.csv, SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json
UPLOAD NEXT: ICHI2027_Phase37_SHAREABLE.json; ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS.csv; ICHI2027_Phase37_NEGATIVE_LINK_TESTS.csv; ICHI2027_Phase37_UNRESOLVED_GATES.csv


### What Phase 37 does **not** demonstrate
- No verified Athena vocabulary, no invented OMOP concept IDs and no applied bindings. Your original **36 vocabulary gap instances remain blocked**.
- No source-attested clinical Procedures were recovered. These 38 fixtures are generated from previously existing *synthetic* ImagingStudies for software tests only.
- No source gender, birthdate, race, ethnicity or clinical MCI has been added.
- No official OMOP or MI-CDM target load, four-arm scientific comparison, clinical outcomes or real ADNI MRI validation.

**Next:** Complete the private Athena snapshot and independent binding reviews, or use these new synthetic Procedure fixtures for isolated SQL cardinality/transaction and idempotence dry-run checks while retaining all vocabulary blockers.